# Eksperimen Deteksi Flaky Test: Stratified K-Fold Cross-Validation

Notebook ini memakai dataset asli FlakeFlagger dari:
- `../test_features.csv`
- `../test_results.csv`

Eksperimen yang dilakukan:
1. **Dataset**: FlakeFlagger features, target `flaky`.
2. **Model**: Random Forest dan XGBoost.
3. **Penanganan Imbalance**: `class_weight='balanced'`, `scale_pos_weight`, SMOTE, Random Undersampling.
4. **Evaluasi Stratified K-Fold**: 
    - 10-Fold CV (90% Train, 10% Test)
    - 5-Fold CV (80% Train, 20% Test)
5. **Thresholding**: threshold dicari di validation set, bukan di test set.
6. **Metrik**: Precision, Recall, F1, Average Precision/PR-AUC, Balanced Accuracy, MCC, dan Confusion Matrix.


In [1]:
# Jika package belum tersedia, jalankan cell ini sekali saja.
# Setelah selesai install, restart kernel lalu Run All.
%pip install pandas numpy matplotlib seaborn scikit-learn xgboost imbalanced-learn nbformat


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    precision_recall_curve,
    matthews_corrcoef,
    roc_auc_score,
)

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler

import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42


## 1. Load Dataset Asli FlakeFlagger

In [3]:
features_path = '../test_features.csv'
results_path = '../test_results.csv'

df_features = pd.read_csv(features_path)
df_results = pd.read_csv(results_path)

print('Shape test_features:', df_features.shape)
print('Shape test_results :', df_results.shape)

print('\nDistribusi label flaky:')
print(df_features['flaky'].value_counts())

display(df_features.head())


Shape test_features: (22236, 29)
Shape test_results : (22245, 8)

Distribusi label flaky:
flaky
0    21425
1      811
Name: count, dtype: int64


,Unnamed: 0,test_name,project,testClassName,testMethodName,flaky,assertion-roulette,conditional-test-logic,eager-test,fire-and-forget,...,projectSourceClassesCovered,hIndexModificationsPerCoveredLine_window5,hIndexModificationsPerCoveredLine_window10,hIndexModificationsPerCoveredLine_window25,hIndexModificationsPerCoveredLine_window50,hIndexModificationsPerCoveredLine_window75,hIndexModificationsPerCoveredLine_window100,hIndexModificationsPerCoveredLine_window500,hIndexModificationsPerCoveredLine_window10000,num_third_party_libs
0,45,ch.qos.logback.classic.asyncappendertest.event...,logback,ch.qos.logback.classic.AsyncAppenderTest,eventWasPreparedForDeferredProcessing,0,1,0,0,0,...,12,0,0,0,0,0,0,1,3,2
1,46,ch.qos.logback.classic.asyncappendertest.setti...,logback,ch.qos.logback.classic.AsyncAppenderTest,settingIncludeCallerDataPropertyCausedCallerDa...,0,1,0,0,0,...,11,0,0,0,0,0,0,1,3,0
2,47,ch.qos.logback.classic.boolex.geventevaluatort...,logback,ch.qos.logback.classic.boolex.GEventEvaluatorTest,callerData,0,0,0,0,0,...,9,0,0,0,0,0,0,1,2,0
3,48,ch.qos.logback.classic.boolex.geventevaluatort...,logback,ch.qos.logback.classic.boolex.GEventEvaluatorTest,event,0,0,0,0,0,...,8,0,0,0,0,0,0,1,2,0
4,49,ch.qos.logback.classic.boolex.geventevaluatort...,logback,ch.qos.logback.classic.boolex.GEventEvaluatorTest,level,0,0,0,0,0,...,10,0,0,0,0,0,0,1,3,0


## 2. Preprocessing

In [4]:
# Label target
y = df_features['flaky'].astype(int)

# Kolom non-feature: identifier/text/label/group
non_feature_cols = [
    'Unnamed: 0',
    'test_name',
    'project',
    'testClassName',
    'testMethodName',
    'flaky',
]

X = df_features.drop(columns=non_feature_cols, errors='ignore')

# Untuk eksperimen awal, gunakan fitur numerik saja.
X = X.select_dtypes(include=[np.number])

# Bersihkan nilai inf dan missing
X = X.replace([np.inf, -np.inf], np.nan).fillna(0)

print('Jumlah sample:', X.shape[0])
print('Jumlah fitur :', X.shape[1])
print('\nDistribusi label:')
print(y.value_counts())


Jumlah sample: 22236
Jumlah fitur : 23

Distribusi label:
flaky
0    21425
1      811
Name: count, dtype: int64


## 3. Fungsi Threshold, Model, dan Evaluasi

In [5]:
def find_optimal_threshold(y_true, y_probs, strategy='max_f1', target_recall=0.80):
    """Cari threshold dari validation set, bukan dari test set."""
    precisions, recalls, thresholds = precision_recall_curve(y_true, y_probs)
    
    if len(thresholds) == 0:
        return 0.5
    
    if strategy == 'max_f1':
        p = precisions[:-1]
        r = recalls[:-1]
        f1_scores = (2 * p * r) / (p + r + 1e-10)
        best_idx = int(np.argmax(f1_scores))
        return float(thresholds[best_idx])
    
    if strategy == 'target_recall':
        valid_idx = np.where(recalls[:-1] >= target_recall)[0]
        if len(valid_idx) == 0:
            return 0.5
        best_idx = int(valid_idx[-1])
        return float(thresholds[best_idx])
    
    return 0.5


def build_models(y_train):
    from sklearn.model_selection import RandomizedSearchCV
    import numpy as np
    
    neg = int(np.sum(y_train == 0))
    pos = int(np.sum(y_train == 1))
    ratio = neg / max(pos, 1)
    
    param_grid = {
        'scale_pos_weight': [1, ratio * 0.5, ratio, ratio * 1.5, ratio * 2],
        'learning_rate': [0.01, 0.05, 0.1],
        'max_depth': [3, 4, 6]
    }
    
    xgb_base = XGBClassifier(
        n_estimators=100,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RANDOM_STATE,
        eval_metric='logloss',
        n_jobs=-1
    )
    
    xgb_tuned = RandomizedSearchCV(
        estimator=xgb_base,
        param_distributions=param_grid,
        n_iter=5,
        scoring='f1',
        cv=3,
        random_state=RANDOM_STATE,
        n_jobs=-1
    )
    
    return {
        'RF_Baseline': RandomForestClassifier(
            n_estimators=200,
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        'RF_Balanced': RandomForestClassifier(
            n_estimators=200,
            class_weight='balanced',
            random_state=RANDOM_STATE,
            n_jobs=-1
        ),
        'RF_SMOTE': ImbPipeline([
            ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=3)),
            ('rf', RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1))
        ]),
        'XGB_ClassWeight_Tuned': xgb_tuned,
        'XGB_RUS': ImbPipeline([
            ('rus', RandomUnderSampler(random_state=RANDOM_STATE)),
            ('xgb', XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.9,
                colsample_bytree=0.9,
                random_state=RANDOM_STATE,
                eval_metric='logloss',
                n_jobs=-1
            ))
        ]),
    }


def safe_roc_auc(y_true, y_probs):
    if len(np.unique(y_true)) < 2:
        return np.nan
    return roc_auc_score(y_true, y_probs)


## 4. Evaluasi Stratified K-Fold
Kita jalankan 2 skema:
- 10-Fold CV (90% Train / 10% Test)
- 5-Fold CV (80% Train / 20% Test)


In [6]:
all_results = []
confusion_records = []

# Fungsi helper untuk menjalankan evaluasi per skema CV
def run_cv_evaluation(n_splits, split_name):
    print(f"\nMenjalankan evaluasi: {split_name} ({n_splits}-Fold CV)...")
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    
    fold = 1
    for train_idx, test_idx in skf.split(X, y):
        print(f"  Memproses Fold {fold}/{n_splits}...")
        X_train_full = X.iloc[train_idx].reset_index(drop=True)
        y_train_full = y.iloc[train_idx].reset_index(drop=True)
        X_test = X.iloc[test_idx].reset_index(drop=True)
        y_test = y.iloc[test_idx].reset_index(drop=True)
        
        # Validation split (20% dari Train) untuk mencari threshold
        X_train, X_val, y_train, y_val = train_test_split(
            X_train_full,
            y_train_full,
            test_size=0.2,
            stratify=y_train_full,
            random_state=RANDOM_STATE
        )
        
        models = build_models(y_train)
        
        for model_name, model in models.items():
            try:
                model.fit(X_train, y_train)
                
                y_val_probs = model.predict_proba(X_val)[:, 1]
                best_thresh = find_optimal_threshold(y_val, y_val_probs, strategy='max_f1')
                
                y_test_probs = model.predict_proba(X_test)[:, 1]
                y_pred = (y_test_probs >= best_thresh).astype(int)
                
                cm = confusion_matrix(y_test, y_pred, labels=[0, 1])
                tn, fp, fn, tp = cm.ravel()
                
                all_results.append({
                    'Skema': split_name,
                    'Fold': fold,
                    'Model': model_name,
                    'Threshold': round(best_thresh, 4),
                    'Test Size': int(len(y_test)),
                    'Flaky in Test': int(y_test.sum()),
                    'Non-Flaky in Test': int((y_test == 0).sum()),
                    'Average Precision / PR-AUC': average_precision_score(y_test, y_test_probs),
                    'ROC-AUC': safe_roc_auc(y_test, y_test_probs),
                    'Precision (Flaky)': precision_score(y_test, y_pred, zero_division=0),
                    'Recall (Flaky)': recall_score(y_test, y_pred, zero_division=0),
                    'F1 (Flaky)': f1_score(y_test, y_pred, zero_division=0),
                    'Balanced Accuracy': balanced_accuracy_score(y_test, y_pred),
                    'MCC': matthews_corrcoef(y_test, y_pred),
                    'TN': int(tn),
                    'FP': int(fp),
                    'FN': int(fn),
                    'TP': int(tp),
                })
                
            except Exception as e:
                print(f'ERROR {split_name} Fold {fold} - {model_name}: {e}')
                
        fold += 1

# 1. 10-Fold CV (90% Train / 10% Test)
run_cv_evaluation(n_splits=10, split_name='90-10 Split (10-Fold)')

# 2. 5-Fold CV (80% Train / 20% Test)
run_cv_evaluation(n_splits=5, split_name='80-20 Split (5-Fold)')

if len(all_results) == 0:
    raise RuntimeError('Tidak ada hasil evaluasi.')

df_results = pd.DataFrame(all_results)
display(df_results.sort_values(by=['Skema', 'Fold', 'F1 (Flaky)'], ascending=[True, True, False]).head(20))



Menjalankan evaluasi: 90-10 Split (10-Fold) (10-Fold CV)...
  Memproses Fold 1/10...
ERROR 90-10 Split (10-Fold) Fold 1 - XGB_ClassWeight_Tuned: Could not pickle the task to send it to the workers.
  Memproses Fold 2/10...
ERROR 90-10 Split (10-Fold) Fold 2 - XGB_ClassWeight_Tuned: Could not pickle the task to send it to the workers.
  Memproses Fold 3/10...
  Memproses Fold 4/10...
  Memproses Fold 5/10...
  Memproses Fold 6/10...
  Memproses Fold 7/10...
  Memproses Fold 8/10...
  Memproses Fold 9/10...
  Memproses Fold 10/10...

Menjalankan evaluasi: 80-20 Split (5-Fold) (5-Fold CV)...
  Memproses Fold 1/5...
  Memproses Fold 2/5...
  Memproses Fold 3/5...
  Memproses Fold 4/5...
  Memproses Fold 5/5...


,Skema,Fold,Model,Threshold,Test Size,Flaky in Test,Non-Flaky in Test,Average Precision / PR-AUC,ROC-AUC,Precision (Flaky),Recall (Flaky),F1 (Flaky),Balanced Accuracy,MCC,TN,FP,FN,TP
49,80-20 Split (5-Fold),1,RF_Balanced,0.5550,4448,163,4285,0.737331,0.961008,0.775362,0.656442,0.710963,0.824604,0.703517,4254,31,56,107
48,80-20 Split (5-Fold),1,RF_Baseline,0.4450,4448,163,4285,0.747907,0.965332,0.827869,0.619632,0.708772,0.807366,0.707182,4264,21,62,101
51,80-20 Split (5-Fold),1,XGB_ClassWeight_Tuned,0.3955,4448,163,4285,0.709762,0.951529,0.844037,0.564417,0.676471,0.780225,0.681080,4268,17,71,92
50,80-20 Split (5-Fold),1,RF_SMOTE,0.5950,4448,163,4285,0.719526,0.958962,0.751880,0.613497,0.675676,0.802898,0.668312,4252,33,63,100
52,80-20 Split (5-Fold),1,XGB_RUS,0.9492,4448,163,4285,0.559311,0.941298,0.759494,0.368098,0.495868,0.681832,0.517327,4266,19,103,60
53,80-20 Split (5-Fold),2,RF_Baseline,0.3250,4447,162,4285,0.801527,0.963162,0.824818,0.697531,0.755853,0.845965,0.750233,4261,24,49,113
54,80-20 Split (5-Fold),2,RF_Balanced,0.4950,4447,162,4285,0.807369,0.970251,0.805970,0.666667,0.729730,0.830299,0.723984,4259,26,54,108
56,80-20 Split (5-Fold),2,XGB_ClassWeight_Tuned,0.7977,4447,162,4285,0.795432,0.963876,0.836066,0.629630,0.718310,0.812481,0.716826,4265,20,60,102
55,80-20 Split (5-Fold),2,RF_SMOTE,0.4700,4447,162,4285,0.776248,0.967553,0.736842,0.691358,0.713376,0.841012,0.703281,4245,40,50,112
57,80-20 Split (5-Fold),2,XGB_RUS,0.9244,4447,162,4285,0.640255,0.949973,0.680851,0.592593,0.633663,0.791045,0.622411,4240,45,66,96


## 5. Ringkasan Hasil

In [7]:
metric_cols = [
    'Average Precision / PR-AUC',
    'ROC-AUC',
    'Precision (Flaky)',
    'Recall (Flaky)',
    'F1 (Flaky)',
    'Balanced Accuracy',
    'MCC'
]

summary_mean = df_results.groupby(['Skema', 'Model'])[metric_cols].mean().reset_index()
summary_mean = summary_mean.sort_values(by=['Skema', 'F1 (Flaky)'], ascending=[True, False])

print('Rata-rata metrik per Skema dan Model:')
display(summary_mean)


Rata-rata metrik per Skema dan Model:


,Skema,Model,Average Precision / PR-AUC,ROC-AUC,Precision (Flaky),Recall (Flaky),F1 (Flaky),Balanced Accuracy,MCC
1,80-20 Split (5-Fold),RF_Baseline,0.778250,0.962707,0.799201,0.658494,0.721585,0.826097,0.715918
0,80-20 Split (5-Fold),RF_Balanced,0.773266,0.967562,0.812521,0.633757,0.710699,0.814055,0.707653
2,80-20 Split (5-Fold),RF_SMOTE,0.750047,0.964908,0.737651,0.675786,0.704660,0.833319,0.695081
3,80-20 Split (5-Fold),XGB_ClassWeight_Tuned,0.748644,0.958395,0.787474,0.617822,0.690206,0.805644,0.686415
4,80-20 Split (5-Fold),XGB_RUS,0.572089,0.945248,0.630959,0.513126,0.556462,0.750449,0.549379
6,90-10 Split (10-Fold),RF_Baseline,0.780808,0.964577,0.788922,0.674496,0.725453,0.833724,0.719155
7,90-10 Split (10-Fold),RF_SMOTE,0.752450,0.965424,0.764330,0.676889,0.716623,0.834431,0.708669
5,90-10 Split (10-Fold),RF_Balanced,0.772095,0.969013,0.743834,0.695333,0.715684,0.842905,0.707272
8,90-10 Split (10-Fold),XGB_ClassWeight_Tuned,0.752793,0.956611,0.820242,0.613162,0.695109,0.803810,0.696428
9,90-10 Split (10-Fold),XGB_RUS,0.623951,0.949120,0.656512,0.528862,0.575679,0.758573,0.570070
